<a href="https://colab.research.google.com/github/pranavtilekar9999/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pranavtilekar9999/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
%pip -q install duckdb

In [2]:
import duckdb
import os
from getpass import getpass

token = os.environ.get("HF_TOKEN")

if not token:
    token = getpass("Enter your Hugging Face READ token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

print("DuckDB connection ready.")

Enter your Hugging Face READ token: ··········
DuckDB connection ready.


In [3]:
FACT = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/'
    'fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FACT}
""").df()

display(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [5]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

display(grain_check)

if grain_check.empty:
    print("PASS: No duplicate date-client-content combinations found.")
else:
    print("Duplicates found. Investigate before proceeding.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


PASS: No duplicate date-client-content combinations found.


In [6]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS ga4_not_confirmed_true_rows
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,ga4_not_confirmed_true_rows
0,9841378,413966,9427412


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
#  Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Lane: Content Refresh / Opportunity Scoring

#Unit of analysis: One content item per client per report date.

# Time window: March 1–31, 2026.

# Features: Historical performance metrics available before the decision moment. Candidate fields will be checked against the warehouse schema and prediction window before use.

# Label/proxy: A future-window performance decline label, to be defined separately from the historical feature window.

# Context `client_hash_id`, `content_hash_id`, and `report_date`. These identify and group records but will not be used as model features.

# Excluded: Future-window metrics and label-derived fields, because they can leak the outcome into the features.

# Availability finding: 413,966 of 9,841,378 March records have `ga4_data_available IS TRUE`. Rows where the flag is `FALSE` or `NULL` are not treated as confirmed GA4-available records.


In [8]:
columns = con.sql(f"""
    DESCRIBE SELECT *
    FROM {FACT}
    LIMIT 1
""").df()

display(columns[["column_name", "column_type"]])

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [9]:
feature_frame = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions,
        ga4_data_available
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
    LIMIT 1000
""").df()

display(feature_frame.head())
print("Feature frame shape:", feature_frame.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>,<NA>


Feature frame shape: (1000, 9)


In [10]:
months_check = con.sql("""
    SELECT 'March 2026' AS period, COUNT(*) AS total_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT 'April 2026' AS period, COUNT(*) AS total_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-04/*.parquet'
    )
""").df()

display(months_check)

,period,total_rows
0,March 2026,9841378
1,April 2026,10424730


In [11]:
march_april_check = con.sql("""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS march_pages,
    COUNT(*) FILTER (
        WHERE april.content_hash_id IS NOT NULL
    ) AS pages_in_both_months,
    COUNT(*) FILTER (
        WHERE april.content_hash_id IS NULL
    ) AS pages_missing_in_april
FROM march
LEFT JOIN april
    ON march.client_hash_id = april.client_hash_id
    AND march.content_hash_id = april.content_hash_id
""").df()

display(march_april_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_in_both_months,pages_missing_in_april
0,176738,158549,18189


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
#  This cell is for CODE (numbers, a query, a check).
#  Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# The warehouse has uneven data availability across clients and dates. In March 2026, 9,841,378 daily records were present, but only 413,966 had `ga4_data_available IS TRUE`. Therefore, GA4-based analysis must explicitly account for availability, and missing or unconfirmed data must not be interpreted as zero engagement. This analysis supports prioritization for human review; it does not prove that refreshing a page will improve its performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.